In [1]:
import numpy as np
import pandas as pd
import os
from PIL import Image
import skimage
from matplotlib import pyplot as plt
from tqdm import tqdm
import cv2
import torch
import pickle
import glob
from skimage.color import label2rgb

import warnings
warnings.filterwarnings('ignore', message='overflow encountered in exp')

In [2]:
# from https://github.com/MouseLand/cellpose/blob/main/notebooks/test_Cellpose-SAM.ipynb

from cellpose import models, core, io, plot, transforms
from pathlib import Path
from tqdm import trange

io.logger_setup() # run this to get printing of progress

#Check if colab notebook instance has GPU access
if core.use_gpu()==False:
  raise ImportError("No GPU access, change your runtime")

model = models.CellposeModel(gpu=True)



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.6 
platform:       	linux 
python version: 	3.13.2 
torch version:  	2.8.0+cu126! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 


2026-04-29 00:34:00,251 [INFO] WRITING LOG OUTPUT TO /home/kerrilu/.cellpose/run.log
2026-04-29 00:34:00,252 [INFO] 
cellpose version: 	4.0.6 
platform:       	linux 
python version: 	3.13.2 
torch version:  	2.8.0+cu126
2026-04-29 00:34:01,435 [INFO] ** TORCH CUDA version installed and working. **
2026-04-29 00:34:01,436 [INFO] ** TORCH CUDA version installed and working. **
2026-04-29 00:34:01,437 [INFO] >>>> using GPU (CUDA)
2026-04-29 00:34:03,948 [INFO] >>>> loading model /home/kerrilu/.cellpose/models/cpsam


In [3]:
def detect_cells(img, T):
    masks, flows, styles = model.eval(img, cellprob_threshold=T)
    return masks, flows, styles

In [4]:
# https://stackoverflow.com/questions/66595055/fastest-way-of-computing-binary-mask-iou-with-numpy
def iou(mask1, mask2):
    intersection = (mask1 * mask2).sum()
    if intersection == 0:
        return 0.0
    union = torch.logical_or(mask1, mask2).to(torch.int).sum()
    return intersection / union

import itertools
def compute_pair_to_iou_dict(masks):
    indices = np.arange(len(masks))
    pairs = itertools.combinations(indices, 2)
    pair_to_iou = {}
    for pair in pairs:
        i, j = pair
        if masks[i].sum()==0 or masks[j].sum()==0:
            pair_to_iou[tuple(pair)] = 0
        else:
            pair_to_iou[tuple(pair)] = iou(masks[i], masks[j]).numpy()
    return pair_to_iou

In [5]:
T_values = np.arange(-5, 5.01, 1)

# Finite sample algorithm

In [6]:
IOU_threshold = 0.75
alpha = .2

In [10]:
file = open('../data/cell_seg/conformal_ious_cellpose_SAM.pkl', 'rb')
results_table = pickle.load(file)
file.close()

In [11]:
m = int(len(results_table)/2)
results_table['calibration_step'] = 'step2'
results_table.loc[results_table.sample(n=m, random_state=0).index, 'calibration_step'] = 'step1'

In [12]:
results_table_1 = results_table[results_table['calibration_step'] == 'step1']
results_table_2 = results_table[results_table['calibration_step'] == 'step2']

In [13]:
print(len(results_table_1))
print(len(results_table_2))

790
790


Step 1

In [14]:
import itertools
import math

T_ordering_indices = []
IOUs = np.array([np.array(x) for x in results_table_1["IOUs"]])
IOUs_high = (IOUs>IOU_threshold).astype(int)
while True:
    coverage_by_T = IOUs_high.sum(axis=0)
    if np.max(coverage_by_T) == 0:
        break
    else:
        best_T = np.argmax(coverage_by_T)
        calibration_pixels_covered = np.argwhere(IOUs_high[:, best_T]==1)
        IOUs_high = np.delete(IOUs_high, calibration_pixels_covered, axis=0)
        T_ordering_indices.append(int(best_T))

T_ordering = T_values[T_ordering_indices]
print("T indices in ranked order from highest to lowest priority:", T_ordering_indices)
print("T in ranked order from highest to lowest priority:", T_ordering)

T indices in ranked order from highest to lowest priority: [5, 3, 7, 4, 0, 9, 2, 6]
T in ranked order from highest to lowest priority: [ 0. -2.  2. -1. -5.  4. -3.  1.]


Step 2

In [18]:
pair_to_iou_dicts = []
classified_as_cell = []
for index, row in tqdm(results_table_2.iterrows(), total=results_table_2.shape[0]):
    image_number = row["image_number"]
    coord = row["coord"]
    pred_shapes_coord = []
    classified_as_cell_coord = []
    
    img = io.imread('../'+image_number)[...,1]
    
    for T in T_ordering[:5]:
        pred_segmentation, _, _ = detect_cells(img, T)
        if pred_segmentation[coord] == 0:
            pred_shape = torch.zeros(img.shape)
            classified_as_cell_coord.append(False)
        else:
            pred_shape = pred_segmentation==pred_segmentation[coord]
            classified_as_cell_coord.append(True)
        pred_shapes_coord.append(torch.from_numpy(pred_shape))
        
    pair_to_iou_dicts.append(compute_pair_to_iou_dict(pred_shapes_coord))
    classified_as_cell.append(classified_as_cell_coord)

results_table_2["Pred Mask Pair to IOUs"] = pair_to_iou_dicts
results_table_2["Classified as Cell"] = classified_as_cell

  1%|█▊                                                                                                                                                              | 9/790 [02:12<3:11:36, 14.72s/it]


KeyboardInterrupt: 

In [17]:
image_number

np.str_('data/cell_seg/cellpose_test_greyscale/044_img.png')